# Enterprise Industrial AI Copilot

# Notebook 03

# OCR Pipeline & Text Corpus Generation

---

## Objective

This notebook extracts text from:

- Images
- Scanned PDFs

using EasyOCR.

Finally it merges OCR text with the native PDF text extracted in Notebook 02 to build one unified enterprise corpus.

Outputs

- ocr_text.parquet
- merged_text_corpus.parquet
- ocr_statistics.json
- ocr_errors.csv

In [1]:
# ==========================================================
# Install Dependencies
# ==========================================================

!pip -q install easyocr pymupdf pillow tqdm


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import fitz
import easyocr
import cv2
import numpy as np
import json
import logging
import os

from PIL import Image

print("Libraries Imported")

Libraries Imported


In [3]:
# ==========================================================
# Universal Notebook Setup
# Compatible with VS Code • GitHub • Local Development
# ==========================================================

import sys
from pathlib import Path

# ----------------------------------------------------------
# Step 1: Automatically locate PROJECT_ROOT
# ----------------------------------------------------------
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = None

# Search upwards (max 5 levels) for src/core/config.py
for _ in range(5):
    if (_current_dir / "src" / "core" / "config.py").is_file():
        PROJECT_ROOT = _current_dir
        break
    _current_dir = _current_dir.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "❌ Could not automatically determine PROJECT_ROOT.\n"\
        "Please make sure the notebook is opened from inside the project."
    )

# ----------------------------------------------------------
# Step 2: Add PROJECT_ROOT to Python path (if needed)
# ----------------------------------------------------------
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ----------------------------------------------------------
# Step 3: Import centralized configuration
# ----------------------------------------------------------
from src.core.config import (
    PROJECT_ROOT as CONFIG_ROOT,
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    VECTOR_DB_DIR,
    KG_DIR,
    INVENTORY_DIR,
    LOG_DIR,OCR_DIR,
)

# ----------------------------------------------------------
# Step 4: Verify PROJECT_ROOT consistency
# ----------------------------------------------------------
if CONFIG_ROOT != PROJECT_ROOT:
    print("⚠ WARNING: Notebook PROJECT_ROOT differs from config.py PROJECT_ROOT")
    print(f"Notebook : {PROJECT_ROOT}")
    print(f"Config   : {CONFIG_ROOT}")

# Use the centralized PROJECT_ROOT from config.py
PROJECT_ROOT = CONFIG_ROOT

# ----------------------------------------------------------
# Step 5: Verify required directories exist
# ----------------------------------------------------------
required_dirs = [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    VECTOR_DB_DIR,
    KG_DIR,
    INVENTORY_DIR,
    LOG_DIR,
]

missing_dirs = [d for d in required_dirs if not d.exists()]

if missing_dirs:
    print("\n❌ Missing Directories:")
    for d in missing_dirs:
        print(f"   - {d}")
    raise FileNotFoundError("One or more required directories are missing.")

# ----------------------------------------------------------
# Step 6: Environment Verification
# ----------------------------------------------------------
print("=" * 65)
print("PROJECT SETUP VERIFICATION SUMMARY")
print("=" * 65)
print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"DATA_DIR          : {DATA_DIR}")
print(f"RAW_DIR           : {RAW_DIR}")
print(f"PROCESSED_DIR     : {PROCESSED_DIR}")
print(f"CHUNK_DIR         : {CHUNK_DIR}")
print(f"EMBEDDING_DIR     : {EMBEDDING_DIR}")
print(f"VECTOR_DB_DIR     : {VECTOR_DB_DIR}")
print(f"KG_DIR            : {KG_DIR}")
print(f"INVENTORY_DIR     : {INVENTORY_DIR}")
print(f"LOG_DIR           : {LOG_DIR}")
print("-" * 65)
print("✅ Status          : SUCCESS")
print("=" * 65)


PROJECT SETUP VERIFICATION SUMMARY
PROJECT_ROOT      : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro
DATA_DIR          : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data
RAW_DIR           : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\raw
PROCESSED_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\processed
CHUNK_DIR         : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\chunks
EMBEDDING_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\embeddings
VECTOR_DB_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\vector_db
KG_DIR            : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\knowledge_graph
INVENTORY_DIR     : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\inventory
LOG_DIR           : C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\logs
-----------------------------------------------------------------
✅ Status          : SUCCESS


In [4]:
# ==========================================================
# Logger
# ==========================================================

logging.basicConfig(

filename=LOG_DIR/"notebook03_ocr.log",

level=logging.INFO,

format="%(asctime)s | %(levelname)s | %(message)s",

force=True

)

logging.info("Notebook03 Started")

print("Logger Ready")

Logger Ready


In [5]:
# ==========================================================
# Load OCR Queue
# ==========================================================

ocr_queue = pd.read_csv(

PROCESSED_DIR/

"ocr_queue.csv"

)

print()

print("="*60)

print("OCR Queue Loaded")

print("="*60)

print()

print("Documents :",len(ocr_queue))


OCR Queue Loaded

Documents : 24


In [6]:
# ==========================================================
# Load Native Text Corpus
# ==========================================================

text_corpus = pd.read_parquet(

PROCESSED_DIR/

"text_documents.parquet"

)

print()

print("Native Pages :",len(text_corpus))


Native Pages : 18459


In [7]:
display(

ocr_queue.head()

)

,Document_ID,File_Name,Absolute_Path,Department,Category,Document_Class
0,DOC00001,86261.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,Operations,OCR Image,IMAGE
1,DOC00002,86797.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,Operations,OCR Image,IMAGE
2,DOC00003,86798.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,Operations,OCR Image,IMAGE
3,DOC00004,86801.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,Operations,OCR Image,IMAGE
4,DOC00005,86808.jpg,C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro...,Operations,OCR Image,IMAGE


In [8]:
# ==========================================================
# Initialize EasyOCR
# ==========================================================

reader = easyocr.Reader(

['en'],

gpu=False

)

print("OCR Engine Ready")

Progress: |--------------------------------------------------| 0.2% Complete

Progress: |--------------------------------------------------| 0.6% Complete

Progress: |--------------------------------------------------| 1.2% Complete

Progress: |--------------------------------------------------| 1.8% Complete

Progress: |█-------------------------------------------------| 2.6% Complete

...

Progress: |█████████████████████████████---------------------| 59.2% Complete

Progress: |██████████████████████████████████████------------| 76.1% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

OCR Engine Ready


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


In [9]:
# ==========================================================
# Supported OCR Formats
# ==========================================================

IMAGE_EXTENSIONS = [

".jpg",

".jpeg",

".png"

]

PDF_EXTENSION = ".pdf"

In [10]:
print("="*60)

print("OCR Pipeline")

print("="*60)

print()

print("Queue :",len(ocr_queue))

print("Native Corpus :",len(text_corpus))

OCR Pipeline

Queue : 24
Native Corpus : 18459


In [11]:
assert len(ocr_queue)>=0

assert len(text_corpus)>0

print("Validation Passed")

Validation Passed


In [12]:
def preprocess_image(image_path):

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(
        gray,
        None,
        fx=2,
        fy=2,
        interpolation=cv2.INTER_CUBIC
    )

    return gray

In [13]:
# ==========================================================
# Robust Image OCR
# ==========================================================

def extract_text_from_image(image_path):
    try:
        image = preprocess_image(image_path)
        if image is None:
            return {
                "Success": False,
                "Text": "",
                "Confidence": 0,
                "Error": f"Image preprocessing returned None for {image_path}"
            }

        results = reader.readtext(
            image,
            detail=1
        )

        extracted_text = []
        confidence_scores = []

        for item in results:
            try:
                # Expecting (bbox, text, prob)
                if not isinstance(item, (list, tuple)) or len(item) != 3:
                    logging.warning(f"Skipping malformed EasyOCR result item (not list/tuple or wrong length): {item}")
                    continue

                bbox, text_content, confidence_score = item

                text = str(text_content).strip()
                conf = float(confidence_score)

                if len(text) == 0:
                    continue

                extracted_text.append(text)
                confidence_scores.append(conf)

            except (TypeError, ValueError, IndexError) as item_error:
                logging.error(f"Error processing individual EasyOCR result item {item}: {item_error}")
                continue # Skip this problematic item and continue with others

        final_text = "\n".join(extracted_text)
        avg_confidence = (
            round(np.mean(confidence_scores),3)
            if confidence_scores else 0
        )

        return {
            "Success": True,
            "Text": final_text,
            "Confidence": avg_confidence
        }

    except Exception as e:
        return {
            "Success": False,
            "Text": "",
            "Confidence": 0,
            "Error": str(e)
        }

In [14]:
sample = reader.readtext(
    str(ocr_queue.iloc[0]["Absolute_Path"]),
    detail=1
)

print(sample)

C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[([[np.int32(43), np.int32(33)], [np.int32(183), np.int32(33)], [np.int32(183), np.int32(53)], [np.int32(43), np.int32(53)]], 'KEEPING YouR', np.float64(0.730471352812928)), ([[np.int32(22), np.int32(54)], [np.int32(208), np.int32(54)], [np.int32(208), np.int32(86)], [np.int32(22), np.int32(86)]], 'WORK AREA', np.float64(0.9756384959283053)), ([[np.int32(62), np.int32(79)], [np.int32(169), np.int32(79)], [np.int32(169), np.int32(109)], [np.int32(62), np.int32(109)]], 'CLEAN', np.float64(0.9999366650894437)), ([[np.int32(61), np.int32(111)], [np.int32(165), np.int32(111)], [np.int32(165), np.int32(131)], [np.int32(61), np.int32(131)]], 'IS PART OF', np.float64(0.6745853384709)), ([[np.int32(41), np.int32(125)], [np.int32(186), np.int32(125)], [np.int32(186), np.int32(157)], [np.int32(41), np.int32(157)]], 'YOUR JOB', np.float64(0.6610943323523576)), ([[np.int32(22), np.int32(150)], [np.int32(204), np.int32(150)], [np.int32(204), np.int32(178)], [np.int32(22), np.int32(178)]], 'ASSIGNMEN

In [15]:
# ==========================================================
# OCR Function for Scanned PDFs
# ==========================================================

def extract_text_from_pdf(pdf_path):

    pages=[]

    try:

        pdf=fitz.open(pdf_path)

        for page in pdf:

            pix=page.get_pixmap(dpi=250)

            image=np.frombuffer(

                pix.samples,

                dtype=np.uint8

            ).reshape(

                pix.height,

                pix.width,

                pix.n

            )

            result=reader.readtext(

                image,

                detail=0,

                paragraph=True

            )

            pages.append(

                "\n".join(result)

            )

        pdf.close()

        return {

            "Success":True,

            "Text":"\n".join(pages)

        }

    except Exception as e:

        return {

            "Success":False,

            "Text":"",

            "Error":str(e)

        }

In [16]:
# ==========================================================
# OCR Entire Queue
# ==========================================================

ocr_results=[]

errors=[]

print("Running OCR...")

Running OCR...


In [17]:
from tqdm.auto import tqdm

for _,row in tqdm(

    ocr_queue.iterrows(),

    total=len(ocr_queue)

):

    path=Path(

        row["Absolute_Path"]

    )

    extension=path.suffix.lower()

    if extension in IMAGE_EXTENSIONS:

        output=extract_text_from_image(path)

    elif extension==".pdf":

        output=extract_text_from_pdf(path)

    else:

        continue

    if output["Success"]:

        ocr_results.append({

            "Document_ID":row["Document_ID"],

            "File_Name":row["File_Name"],

            "Department":row["Department"],

            "Category":row["Category"],

            "OCR_Text":output["Text"],

            "Confidence":

                output.get(

                    "Confidence",

                    None

                )

        })

    else:

        errors.append({

            "Document_ID":row["Document_ID"],

            "File_Name":row["File_Name"],

            "Error":output["Error"]

        })

  0%|          | 0/24 [00:00<?, ?it/s]

C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [18]:
ocr_df=pd.DataFrame(

ocr_results

)

display(

ocr_df.head()

)

,Document_ID,File_Name,Department,Category,OCR_Text,Confidence
0,DOC00001,86261.jpg,Operations,OCR Image,KEEPING YOUR\nWORK AREA\nCLEAN\nIS PART OF\nYO...,0.764
1,DOC00002,86797.jpg,Operations,OCR Image,DANGER\nDo not operate\nwithout guards|\nin pl...,0.650
2,DOC00003,86798.jpg,Operations,OCR Image,DANGER\nEar protection\nrequired while\noperat...,0.763
3,DOC00004,86801.jpg,Operations,OCR Image,DANGER\nFlammable.\nDNLY: tai [4io i,0.529
4,DOC00005,86808.jpg,Operations,OCR Image,"DANGER\nHot,\nDD"" 1ai Iioa=",0.474


In [19]:
errors_df=pd.DataFrame(

errors

)

print()

print("="*60)

print("OCR Completed")

print("="*60)

print()

print("Success :",len(ocr_df))

print("Errors :",len(errors_df))


OCR Completed

Success : 24
Errors : 0


In [20]:
# ==========================================================
# Basic Cleaning
# ==========================================================

def clean_text(text):

    if pd.isna(text):

        return ""

    text=text.replace("\n\n","\n")

    text=text.replace("\t"," ")

    text=text.replace("  "," ")

    return text.strip()

## Merge OCR Text with Native Text Corpus

Now that we have both OCR extracted text (`ocr_df`) and native PDF text (`text_corpus`), we will merge them into a single, unified enterprise corpus. Before merging, we will apply the `clean_text` function to the OCR text and standardize the columns for concatenation.

In [21]:
# Apply the clean_text function to the OCR_Text column
ocr_df["OCR_Text"] = ocr_df["OCR_Text"].apply(clean_text)

# Prepare ocr_df for merging by selecting relevant columns and renaming
ocr_corpus_df = ocr_df[[
    "Document_ID",
    "File_Name",
    "OCR_Text"
]].copy()
ocr_corpus_df.rename(
    columns={"OCR_Text": "text"},
    inplace=True
)
ocr_corpus_df["source"] = "OCR"
ocr_corpus_df["page_number"] = None # OCR text is per document, not per page

In [22]:
# Prepare text_corpus for merging by selecting relevant columns and renaming
print("text_corpus columns before selection:", text_corpus.columns)
native_corpus_df = text_corpus[[
    "Document_ID",
    "File_Name",
    "Page_Number",
    "Text"
]].copy()
native_corpus_df.rename(
    columns={"Text": "text"},
    inplace=True
)
native_corpus_df["source"] = "Native PDF"

text_corpus columns before selection: Index(['Document_ID', 'File_Name', 'Page_Number', 'Text'], dtype='str')


In [23]:
# Concatenate both dataframes to form the unified corpus
merged_corpus = pd.concat(
    [ocr_corpus_df, native_corpus_df],
    ignore_index=True
)

print("Unified Corpus Created!")
display(merged_corpus.head())

Unified Corpus Created!


,Document_ID,File_Name,text,source,page_number,Page_Number
0,DOC00001,86261.jpg,KEEPING YOUR\nWORK AREA\nCLEAN\nIS PART OF\nYO...,OCR,None,NaN
1,DOC00002,86797.jpg,DANGER\nDo not operate\nwithout guards|\nin pl...,OCR,None,NaN
2,DOC00003,86798.jpg,DANGER\nEar protection\nrequired while\noperat...,OCR,None,NaN
3,DOC00004,86801.jpg,DANGER\nFlammable.\nDNLY: tai [4io i,OCR,None,NaN
4,DOC00005,86808.jpg,"DANGER\nHot,\nDD"" 1ai Iioa=",OCR,None,NaN


In [24]:
errors_df.head(20)

""


In [25]:
display(errors_df)

""


In [26]:
print(errors_df.shape)

(0, 0)


In [27]:
errors_df = pd.DataFrame(
    errors,
    columns=[
        "Document_ID",
        "File_Name",
        "Error"
    ]
)

In [28]:
print("OCR Success:", len(ocr_df))
print("OCR Errors :", len(errors_df))
print("Merged Corpus:", len(merged_corpus))

OCR Success: 24
OCR Errors : 0
Merged Corpus: 18483


In [29]:
print(len(merged_corpus))

18483


In [30]:
# ==========================================================
# Export OCR Text
# ==========================================================

ocr_output = OCR_DIR / "ocr_text.parquet"

ocr_df.to_parquet(
    ocr_output,
    index=False
)

print(f"OCR Text Saved:\n{ocr_output}")

OCR Text Saved:
C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\ocr\ocr_text.parquet


In [31]:
# ==========================================================
# Export Merged Corpus
# ==========================================================

merged_output = OCR_DIR / "merged_text_corpus.parquet"

merged_corpus.to_parquet(
    merged_output,
    index=False
)

print(f"Merged Corpus Saved:\n{merged_output}")

Merged Corpus Saved:
C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\ocr\merged_text_corpus.parquet


In [32]:
# ==========================================================
# Export OCR Errors
# ==========================================================

error_output = OCR_DIR / "ocr_errors.csv"

errors_df.to_csv(
    error_output,
    index=False
)

print(f"Errors Saved:\n{error_output}")

Errors Saved:
C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\ocr\ocr_errors.csv


In [33]:
# ==========================================================
# OCR Statistics
# ==========================================================

ocr_statistics = {

    "Total OCR Documents": int(len(ocr_queue)),

    "OCR Success": int(len(ocr_df)),

    "OCR Errors": int(len(errors_df)),

    "Merged Corpus Records": int(len(merged_corpus)),

    "Unique Documents": int(
        merged_corpus["Document_ID"].nunique()
    ),

    "OCR Images": int(
        (ocr_queue["Document_Class"]=="IMAGE").sum()
    ),

    "Scanned PDFs": int(
        (ocr_queue["Document_Class"]=="SCANNED_PDF").sum()
    )

}

ocr_statistics

{'Total OCR Documents': 24,
 'OCR Success': 24,
 'OCR Errors': 0,
 'Merged Corpus Records': 18483,
 'Unique Documents': 59,
 'OCR Images': 24,
 'Scanned PDFs': 0}

In [34]:
# ==========================================================
# Export Statistics
# ==========================================================

statistics_output = OCR_DIR / "ocr_statistics.json"

with open(statistics_output,"w") as f:

    json.dump(

        ocr_statistics,

        f,

        indent=4

    )

print(statistics_output)

C:\Users\LENOVO\OneDrive\Desktop\Hackathon_pro\data\ocr\ocr_statistics.json


In [35]:
# ==========================================================
# Statistics Dashboard
# ==========================================================

print("="*70)

print("OCR PIPELINE SUMMARY")

print("="*70)

for k,v in ocr_statistics.items():

    print(f"{k:<30}: {v}")

print("="*70)

OCR PIPELINE SUMMARY
Total OCR Documents           : 24
OCR Success                   : 24
OCR Errors                    : 0
Merged Corpus Records         : 18483
Unique Documents              : 59
OCR Images                    : 24
Scanned PDFs                  : 0


In [36]:
# ==========================================================
# Output Verification
# ==========================================================

outputs=[

ocr_output,

merged_output,

error_output,

statistics_output

]

print("="*60)

print("Generated Files")

print("="*60)

for file in outputs:

    print(

        f"{file.name:<35}",

        file.exists()

    )

Generated Files
ocr_text.parquet                    True
merged_text_corpus.parquet          True
ocr_errors.csv                      True
ocr_statistics.json                 True


In [37]:
# ==========================================================
# Final Validation
# ==========================================================

assert ocr_output.exists()

assert merged_output.exists()

assert error_output.exists()

assert statistics_output.exists()

assert len(merged_corpus)>0

assert len(ocr_df)>0

print("Notebook 03 Validation Passed")

Notebook 03 Validation Passed


In [38]:
# ==========================================================
# Sample OCR Output
# ==========================================================

display(

ocr_df[

[

"Document_ID",

"File_Name",

"Confidence",

"OCR_Text"

]

].head()

)

,Document_ID,File_Name,Confidence,OCR_Text
0,DOC00001,86261.jpg,0.764,KEEPING YOUR\nWORK AREA\nCLEAN\nIS PART OF\nYO...
1,DOC00002,86797.jpg,0.650,DANGER\nDo not operate\nwithout guards|\nin pl...
2,DOC00003,86798.jpg,0.763,DANGER\nEar protection\nrequired while\noperat...
3,DOC00004,86801.jpg,0.529,DANGER\nFlammable.\nDNLY: tai [4io i
4,DOC00005,86808.jpg,0.474,"DANGER\nHot,\nDD"" 1ai Iioa="


In [39]:
# ==========================================================
# Sample Merged Corpus
# ==========================================================

display(

merged_corpus.head(10)

)

,Document_ID,File_Name,text,source,page_number,Page_Number
0,DOC00001,86261.jpg,KEEPING YOUR\nWORK AREA\nCLEAN\nIS PART OF\nYO...,OCR,None,NaN
1,DOC00002,86797.jpg,DANGER\nDo not operate\nwithout guards|\nin pl...,OCR,None,NaN
2,DOC00003,86798.jpg,DANGER\nEar protection\nrequired while\noperat...,OCR,None,NaN
3,DOC00004,86801.jpg,DANGER\nFlammable.\nDNLY: tai [4io i,OCR,None,NaN
4,DOC00005,86808.jpg,"DANGER\nHot,\nDD"" 1ai Iioa=",OCR,None,NaN
5,DOC00006,86839.jpg,WARNING\nWatch your\nhands\nand\ntingers:\nDNY...,OCR,None,NaN
6,DOC00007,86860.jpg,DANGER\nHigh\nvoltage\nDNY 1839 Ilioer,OCR,None,NaN
7,DOC00008,86861.jpg,DANGER\nHigh\nvoltage\ninside.\nDNY Io1 1lios,OCR,None,NaN
8,DOC00009,86868.jpg,DANGER\nPinch\nDNY 1844 14109192\npoint;,OCR,None,NaN
9,DOC00010,86877.jpg,DANGER\nHigh voltage:\nEntry by authorized\npe...,OCR,None,NaN


In [40]:
# ==========================================================
# Notebook Completion
# ==========================================================

logging.info("="*80)

logging.info("Notebook 03 Completed Successfully")

logging.info("="*80)

print()

print("="*80)

print("NOTEBOOK 03 COMPLETED SUCCESSFULLY")

print("="*80)


NOTEBOOK 03 COMPLETED SUCCESSFULLY


In [41]:
print("""

Next Notebook

Notebook 04

Semantic Chunking
+
Embeddings
+
FAISS Index

Input

merged_text_corpus.parquet

Output

chunks.parquet

embeddings.npy

faiss.index

""")



Next Notebook

Notebook 04

Semantic Chunking
+
Embeddings
+
FAISS Index

Input

merged_text_corpus.parquet

Output

chunks.parquet

embeddings.npy

faiss.index


